## Preparation

In [29]:
import torch
# from transformers.activations import ACT2FN
from transformer_lens import HookedTransformer
import examples
from neuron_analysis import neuron_analysis

In [ ]:
# import importlib
# importlib.reload(examples)

In [2]:
DEVICE='cuda:0'

In [3]:
args = examples.create_args(
    neuron_subset_name="weakening",
    intervention_type="zero_ablation",
    metric="entropy",
    topk=1,
    device=DEVICE,
    gate='-',
    post='+',
    #data_dir="../RW_functionalities_results",
)

In [4]:
model = HookedTransformer.from_pretrained(
    'allenai/OLMo-7B-0424-hf',
    #refactor_glu=True,
    device=DEVICE
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded pretrained model allenai/OLMo-7B-0424-hf into HookedTransformer


In [5]:
input_ids = model.to_tokens(
    "Yesterday (21 December) the Government announced a package of support for hospitality and leisure businesses that are losing trade because of the Omic"
)#.flatten()

In [6]:
neuron_list = examples.find_neurons(args)
print(neuron_list)

[(tensor(0, device='cuda:0'), tensor(365, device='cuda:0')), (tensor(0, device='cuda:0'), tensor(3234, device='cuda:0')), (tensor(0, device='cuda:0'), tensor(3467, device='cuda:0')), (tensor(0, device='cuda:0'), tensor(5769, device='cuda:0')), (tensor(0, device='cuda:0'), tensor(10043, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(130, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(227, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(1539, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(1720, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(2192, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(2283, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(2328, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(2638, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(3983, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(3985, device='cuda:0')), (tensor(1, device='cuda:0'), tensor(4101, device='cuda:0')), (tensor(1, device='cuda:0

In [7]:
result_str, cache_clean, cache_ablated = examples.show_single_text(
    args=args,
    model=model,
    input_ids=input_ids,
    pos=input_ids.shape[1]-2,#ignore the last token, which is the ground-truth output
    neuron_list=neuron_list,
    return_cache=True,
)

running clean model...
dict_keys(['hook_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_rot_q', 'blocks.0.attn.hook_rot_k', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_pre_linear', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_rot_q', 'blocks.1.attn.hook_rot_k', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_n

In [8]:
print(result_str)

Input tokens:
["<|endoftext|>", "Yesterday", " (", "21", " December", ")", " the", " Government", " announced", " a", " package", " of", " support", " for", " hospitality", " and", " leisure", " businesses", " that", " are", " losing", " trade", " because", " of", " the", " O", "mic"]
Ground-truth output token:
mic

The clean model outputs:
mic
with score 21.1351261138916

The ablated model outputs:


with score 9.471395492553711
top tokens promoted by the neurons (overall effect):
["mic", "MIC", "Mic", "MI", "micro", "Micro", " Mic", "PEC", "NS", "mi", " mic", "yster", "ECD", "mn", "VO", "microm"]
top tokens suppressed by the neurons (overall effect):
[" the", " one", " old", " a", " older", " many", " individual", " make", " it", " them", " our", " their", " big", " others", " other", " early"]


In [9]:
(model.ln_final(cache_clean['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

21.135128021240234

In [10]:
(model.ln_final(cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

1.3977943658828735

score difference to explain, *without* layer norm:

In [11]:
((cache_clean['blocks.31.hook_resid_post'][0,-1]-cache_ablated['blocks.31.hook_resid_post'][0,-1])@model.W_U.detach()[:,model.to_single_token('mic')]).item()

3.579124927520752

Note: I suspect llmtt computes the scores *with* layer norm, hence a mismatch.

## Neuron 31.2578
we already know from llmtt this neuron is relevant.

In [13]:
neuron_analysis(model, 31, 2578)

gate vs. linear similarity: 0.19408632814884186
gate vs. out similarity: -0.4562482535839081
lin vs. out similarity: -0.41817840933799744
most similar tokens for w_out:
        dot product
Kem        0.020921
Newman     0.016358
Dodge      0.015971
Salem      0.014772
Fors       0.014623
Pho        0.014424
Mang       0.013753
Ri         0.013726
Karn       0.013583
Prest      0.013499
most similar tokens for -w_out:
              dot product
 micro           0.185669
 Micro           0.172217
micro            0.158704
 mic             0.148140
 Mic             0.146873
Micro            0.145346
mic              0.131744
Mic              0.121577
 microm          0.116042
 microscopic     0.104223
most similar tokens for w_in:
             dot product
 micro          0.136597
 Micro          0.119756
micro           0.092803
Micro           0.092235
 mic            0.075419
 Mic            0.069867
 microp         0.062157
 microm         0.059119
Mic             0.055986
 microphone  

In [14]:
print("Re-running clean model...")
logits_clean_new, cache_clean_new = model.run_with_cache(input_ids[:,:input_ids.shape[1]-1])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

Re-running clean model...
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


How is w_in related to 'mic'?

In [15]:
model.blocks[31].mlp.W_in.detach()[:,2578]@model.W_U.detach()[:,model.to_single_token('mic')]

tensor(0.0441, device='cuda:0')

when activated positively, gate and in both detect 'mic' and write minus 'mic' (next to other similar things)

In [16]:
for subkey in ['pre', 'pre_linear', 'post']:
    print(subkey)
    print('clean')
    print(cache_clean[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())#batch pos neuron
    print('ablated')
    print(cache_ablated[f'blocks.31.mlp.hook_{subkey}'][0,-1,2578].item())
    print('\n')

pre
clean
5.226315498352051
ablated
0.5835620760917664


pre_linear
clean
-1.5932304859161377
ablated
-0.03865267336368561


post
clean
-8.282222747802734
ablated
-0.01447854470461607




In the clean run, the gate detects 'mic' but 'in' does not, leading to a strengthening of 'mic'. In the ablated run, activations are in the same quadrant but much smaller.

In the clean run compared to the ablated run, the neuron increases the mic logit by approx. 8.28*0.443 = ...

In [17]:
(
    cache_clean['blocks.31.mlp.hook_post'][0,-1,2578] - cache_ablated['blocks.31.mlp.hook_post'][0,-1,2578]
).item() * (
    model.blocks[31].mlp.W_out[2578,:].detach()@(model.W_U.detach()[:,model.to_single_token('mic')])
).item()

1.0892278695630466

This *does not* completely explain the score difference of 3.579124927520752.

What about the residual stream before the last MLP?

In [18]:
(
    (
    cache_clean['blocks.31.hook_resid_mid'][0,-1] - cache_ablated['blocks.31.hook_resid_mid'][0,-1]
    ) @ (
        model.W_U.detach()[:,model.to_single_token('mic')]
    )
).item()

0.7509428262710571

What about the output of the MLP layer as a whole (without residual connection)?

In [19]:
(
    (
    cache_clean['blocks.31.hook_mlp_out'][0,-1] - cache_ablated['blocks.31.hook_mlp_out'][0,-1]
    ) @ (
        model.W_U.detach()[:,model.to_single_token('mic')]
    )
).item()

2.8281819820404053

What about the aggregate output of the ablated neurons in the MLP layer? (i.e., ignoring indirect effects from earlier layers)

In [42]:
#hacky: we check which activations were set to exact zeros - this doesn't work with other ablation types
indices = torch.where(cache_ablated['blocks.31.mlp.hook_post'][0,-1]==0)[0]
print(indices)

tensor([  738,  2665,  3133,  4192,  5751,  6228,  6831,  6885,  7685,  8845,
        10244], device='cuda:0')


In [44]:
(
    (
        cache_clean['blocks.31.mlp.hook_post'][0,-1, indices] - cache_ablated['blocks.31.mlp.hook_post'][0,-1, indices]
    )
    @ (
        model.blocks[31].mlp.W_out[indices,:]
    )
    @ (
        model.W_U.detach()[:,model.to_single_token('mic')]
    )
).item()

-0.0014831344597041607

Correction: the final MLP layer plays a big role, but that is not fully explained by the 'mic' neuron, and even the whole layer doesn't explain everything.
Also, the direct effect of the ablated neurons is even slightly negative.

Next big question: What causes this neuron to activate in one run and not the other?

## Going further back

### Debugging

In [20]:
for layer in range(32):
    for hook in ["hook_resid_pre", "hook_resid_mid", "ln2.hook_normalized", "mlp.hook_pre", "mlp.hook_pre_linear", "mlp.hook_post", "hook_mlp_out", "hook_resid_post"]:
        diff = (
            cache_clean_new[f"blocks.{layer}.{hook}"][0:1,-1:] - cache_clean[f"blocks.{layer}.{hook}"][0:1,-1:]
        ).abs().max().item()
        if diff!=0:
            print(layer, hook)
            print(diff)

In [21]:
print(model.blocks[31].apply_mlp(cache_clean[f'blocks.31.ln2.hook_normalized'][0:1,-1:]))
print(cache_clean[f'blocks.31.hook_mlp_out'][0:1,-1:])
print(cache_clean_new[f'blocks.31.hook_mlp_out'][0:1,-1:])

tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


In [ ]:
# x = cache_clean_new["blocks.31.ln2.hook_normalized"][0:1, -1:]
# y = model.blocks[31].apply_mlp(x)
# # print("x shape", x.shape)
# # print("y shape", y.shape)
# # print("cached shape", cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:].shape)
# print("allclose full slice", torch.allclose(y, cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:], atol=1e-6, rtol=1e-4))
# print("max abs diff full slice", (y - cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:]).abs().max().item())
# y1 = model.blocks[31].apply_mlp(cache_clean_new["blocks.31.ln2.hook_normalized"][0,-1])
# # print("y1 shape", y1.shape)
# print("allclose 1d vs cached 1d", torch.allclose(y1, cache_clean_new["blocks.31.hook_mlp_out"][0,-1], atol=1e-6, rtol=1e-4))
# print("max abs diff 1d", (y1 - cache_clean_new["blocks.31.hook_mlp_out"][0,-1]).abs().max().item())


NameError: name 'torch' is not defined

In [ ]:
# print(y)
# print(y1)
# print(cache_clean_new["blocks.31.hook_mlp_out"][0:1,-1:])

tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')
tensor([ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698],
       device='cuda:0')
tensor([[[ 0.0374, -0.0318, -0.2299,  ..., -0.1227,  0.1623, -0.1698]]],
       device='cuda:0')


In [22]:
model.hook_dict

{'hook_embed': HookPoint(),
 'blocks.0.ln1.hook_scale': HookPoint(),
 'blocks.0.ln1.hook_normalized': HookPoint(),
 'blocks.0.ln2.hook_scale': HookPoint(),
 'blocks.0.ln2.hook_normalized': HookPoint(),
 'blocks.0.attn.hook_k': HookPoint(),
 'blocks.0.attn.hook_q': HookPoint(),
 'blocks.0.attn.hook_v': HookPoint(),
 'blocks.0.attn.hook_z': HookPoint(),
 'blocks.0.attn.hook_attn_scores': HookPoint(),
 'blocks.0.attn.hook_pattern': HookPoint(),
 'blocks.0.attn.hook_result': HookPoint(),
 'blocks.0.attn.hook_rot_k': HookPoint(),
 'blocks.0.attn.hook_rot_q': HookPoint(),
 'blocks.0.mlp.hook_pre': HookPoint(),
 'blocks.0.mlp.hook_pre_linear': HookPoint(),
 'blocks.0.mlp.hook_post': HookPoint(),
 'blocks.0.hook_attn_in': HookPoint(),
 'blocks.0.hook_q_input': HookPoint(),
 'blocks.0.hook_k_input': HookPoint(),
 'blocks.0.hook_v_input': HookPoint(),
 'blocks.0.hook_mlp_in': HookPoint(),
 'blocks.0.hook_attn_out': HookPoint(),
 'blocks.0.hook_mlp_out': HookPoint(),
 'blocks.0.hook_resid_pre': H

In [20]:
model.blocks[31].mlp.b_in[2578]

tensor(0., device='cuda:7', requires_grad=True)

In [26]:
model.blocks[31].ln2

LayerNormPre(
  (hook_scale): HookPoint()
  (hook_normalized): HookPoint()
)

In [30]:
# def hypothetical_neuron_activation(residual):
#         #normalised_residual = residual
#         normalised_residual = model.blocks[31].ln2(residual)
#         return (ACT2FN[model.cfg.act_fn](normalised_residual @ model.blocks[31].mlp.W_gate[:,2578]) * (normalised_residual @ model.blocks[31].mlp.W_in[:,2578])).item()

def hypothetical_ln(residual):
    return model.blocks[31].ln2(residual)

def hypothetical_hook_pre(x):
    return x @ model.blocks[31].mlp.W_gate[:,2578]

def hypothetical_hook_pre_linear(x):
    return x @ model.blocks[31].mlp.W_in[:,2578]

def act_from_branches(pre_act, pre_linear):
    ans = model.blocks[31].mlp.act_fn(pre_act) * pre_linear + model.blocks[31].mlp.b_in[2578]
    if isinstance(ans, torch.Tensor):
        return ans.item()
    else:
        return ans

def hypothetical_neuron_activation(x, verbose=True, with_ln=True):
    if with_ln:
        x = hypothetical_ln(x)
    pre_act = hypothetical_hook_pre(x)
    pre_linear = hypothetical_hook_pre_linear(x)
    post = act_from_branches(pre_act=pre_act, pre_linear=pre_linear)
    if verbose:
        print(f'>> pre: {pre_act}')
        print(f'>> pre_linear: {pre_linear}')
        print(f'>> post: {post}')
    return post


In [31]:
def show_hypothetical_activations(layer, loc, **kwargs):
    print('> clean:')
    hypothetical_clean_activation = hypothetical_neuron_activation(cache_clean[f'blocks.{layer}.hook_resid_{loc}'][0,-1], **kwargs)
    print('> ablated:')
    hypothetical_ablated_activation = hypothetical_neuron_activation(cache_ablated[f'blocks.{layer}.hook_resid_{loc}'][0,-1], **kwargs)
    print("> diff:", hypothetical_clean_activation - hypothetical_ablated_activation)

In [26]:
print(
    "originally computed normalised residual:", cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1]
)
print(
    "recomputed:", hypothetical_ln(cache_clean['blocks.31.hook_resid_mid'][0,-1])
)

originally computed normalised residual: tensor([-1.0645,  1.1939, -0.0757,  ...,  0.4562, -1.2905,  0.7178],
       device='cuda:0')
recomputed: tensor([-1.0645,  1.1939, -0.0757,  ...,  0.4562, -1.2905,  0.7178],
       device='cuda:0')


In [27]:
(cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1]-hypothetical_ln(cache_clean['blocks.31.hook_resid_mid'][0,-1])).abs().max()

tensor(7.4506e-09, device='cuda:0')

In [32]:
act_from_branches(
    pre_act=cache_clean[f'blocks.31.mlp.hook_pre'][0,-1,2578],
    pre_linear=cache_clean[f'blocks.31.mlp.hook_pre_linear'][0,-1,2578],
)

-8.282222747802734

In [33]:
hypothetical_hook_pre(cache_clean[f'blocks.31.ln2.hook_normalized'][0,-1])

tensor(5.2263, device='cuda:0')

In [42]:
model.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'silu',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(11.313708498984761),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 128,
 'd_mlp': 11008,
 'd_model': 4096,
 'd_vocab': 50304,
 'd_vocab_out': 50304,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda:7',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': True,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.0125),
 'load_in_4bit': False,
 'model_name': 'OLMo-7B-0424-hf',
 'n_ctx': 4096,
 'n_devices': 1,
 'n_heads': 32,
 'n_key_value_heads': None,
 'n_layers': 32,
 'n_params': 6476005376,
 'norma

In [43]:
model.cfg.is_layer_norm_activation()

False

In [37]:
model.blocks

ModuleList(
  (0-31): 32 x TransformerBlock(
    (ln1): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (ln2): LayerNormPre(
      (hook_scale): HookPoint()
      (hook_normalized): HookPoint()
    )
    (attn): Attention(
      (hook_k): HookPoint()
      (hook_q): HookPoint()
      (hook_v): HookPoint()
      (hook_z): HookPoint()
      (hook_attn_scores): HookPoint()
      (hook_pattern): HookPoint()
      (hook_result): HookPoint()
      (hook_rot_k): HookPoint()
      (hook_rot_q): HookPoint()
    )
    (mlp): GatedMLP(
      (hook_pre): HookPoint()
      (hook_pre_linear): HookPoint()
      (hook_post): HookPoint()
    )
    (hook_attn_in): HookPoint()
    (hook_q_input): HookPoint()
    (hook_k_input): HookPoint()
    (hook_v_input): HookPoint()
    (hook_mlp_in): HookPoint()
    (hook_attn_out): HookPoint()
    (hook_mlp_out): HookPoint()
    (hook_resid_pre): HookPoint()
    (hook_resid_mid): HookPoint()
    (hook_resid_post): HookP

In [35]:
print("with ln:")
show_hypothetical_activations(31, "mid", with_ln=True)
print("without ln:")
show_hypothetical_activations(31, "mid", with_ln=False)

with ln:
> clean:
>> pre: 5.226315498352051
>> pre_linear: -1.5932306051254272
>> post: -8.28222370147705
> ablated:
>> pre: 0.5835621953010559
>> pre_linear: -0.038652800023555756
>> post: -0.014478596858680248
> diff: -8.26774510461837
without ln:
> clean:
>> pre: 0.7053793668746948
>> pre_linear: -0.21503330767154694
>> post: -0.1015314981341362
> ablated:
>> pre: 0.6305511593818665
>> pre_linear: -0.041765160858631134
>> post: -0.017186647281050682
> diff: -0.08434485085308552


### Actual continuation

In [36]:
for layer in reversed(range(32)):
    for loc in ["mid", "pre"]:
        print(f"layer {layer}, {loc}:")
        print(
            (
                (
                cache_clean[f'blocks.{layer}.hook_resid_{loc}'][0,-1] - cache_ablated[f'blocks.{layer}.hook_resid_{loc}'][0,-1]
                ) @ (
                    model.W_U.detach()[:,model.to_single_token('mic')]
                )
            ).item()
        )

layer 31, mid:
0.7509428262710571
layer 31, pre:
0.1944030523300171
layer 30, mid:
0.14591200649738312
layer 30, pre:
0.1937163770198822
layer 29, mid:
0.14523379504680634
layer 29, pre:
0.16407600045204163
layer 28, mid:
0.1325215995311737
layer 28, pre:
0.13410057127475739
layer 27, mid:
0.1370631903409958
layer 27, pre:
0.1343434900045395
layer 26, mid:
0.13434049487113953
layer 26, pre:
0.1287311315536499
layer 25, mid:
0.10406903177499771
layer 25, pre:
0.1045655906200409
layer 24, mid:
0.09616781771183014
layer 24, pre:
0.09737484157085419
layer 23, mid:
0.09885534644126892
layer 23, pre:
0.09779899567365646
layer 22, mid:
0.09885097295045853
layer 22, pre:
0.09911633282899857
layer 21, mid:
0.09976546466350555
layer 21, pre:
0.10003344714641571
layer 20, mid:
0.12407959252595901
layer 20, pre:
0.10495232045650482
layer 19, mid:
0.13854050636291504
layer 19, pre:
0.14299266040325165
layer 18, mid:
0.15510928630828857
layer 18, pre:
0.14929857850074768
layer 17, mid:
0.16543108224

In [37]:
for layer in reversed(range(32)):
    for loc in ["mid", "pre"]:
        print(f"layer {layer}, {loc}:")
        show_hypothetical_activations(layer, loc)

layer 31, mid:
> clean:
>> pre: 5.226315498352051
>> pre_linear: -1.5932306051254272
>> post: -8.28222370147705
> ablated:
>> pre: 0.5835621953010559
>> pre_linear: -0.038652800023555756
>> post: -0.014478596858680248
> diff: -8.26774510461837
layer 31, pre:
> clean:
>> pre: 4.93642520904541
>> pre_linear: -1.4987211227416992
>> post: -7.345581531524658
> ablated:
>> pre: 4.465673446655273
>> pre_linear: -0.42207223176956177
>> post: -1.8634132146835327
> diff: -5.4821683168411255
layer 30, mid:
> clean:
>> pre: 3.466919422149658
>> pre_linear: -1.139691710472107
>> post: -3.83162260055542
> ablated:
>> pre: 3.5029263496398926
>> pre_linear: -1.097286581993103
>> post: -3.73136568069458
> diff: -0.10025691986083984
layer 30, pre:
> clean:
>> pre: 3.506455659866333
>> pre_linear: -1.3010566234588623
>> post: -4.4292073249816895
> ablated:
>> pre: 3.6522557735443115
>> pre_linear: -0.5058950185775757
>> post: -1.8009545803070068
> diff: -2.6282527446746826
layer 29, mid:
> clean:
>> pre:

We need automation here. Goal: nested dictionary. The keys should be names of units. For each key, the value should show:

* the effect of the difference in activations of that unit on the metric (here, score of "mic"), via the unit corresponding to the outer dictionary
* if applicable, the effect of ablated neurons *in that unit* via the same path
* similar dictionaries for the directly preceding units (previous residual stream, previous unit)

Alternative: edge attribution patching in its simplest form: activation of earlier unit (a d_model vector) times gradient of metric wrt the later unit (also a d_model vector). But:

* It still doesn't make sense to me
* could it suffer from adversarial effects like the memory management paper?

To check: is my method actually the same as...
* ACDC -> no, ACDC applies activation patching
* causal scrubbing? -> no, this is an evaluation method